<div align="center">
    <span style="font-size:30px">
        <strong>
            <!-- Símbolo de Python -->
            <img
                src="https://cdn3.emoji.gg/emojis/1887_python.png"
                style="margin-bottom:-5px"
                width="30px" 
                height="30px"
            >
            <!-- Título -->
            Python para Geólogos
            <!-- Versión -->
            <img 
                src="https://img.shields.io/github/release/kevinalexandr19/manual-python-geologia.svg?style=flat&label=&color=blue"
                style="margin-bottom:-2px" 
                width="40px"
            >
        </strong>
    </span>
    <br>
    <span>
        <!-- Github del proyecto -->
        <a href="https://github.com/kevinalexandr19/manual-python-geologia" target="_blank">
            <img src="https://img.shields.io/github/stars/kevinalexandr19/manual-python-geologia.svg?style=social&label=Github Repo">
        </a>
        &nbsp;&nbsp;
        <!-- Licencia -->
        <img src="https://img.shields.io/github/license/kevinalexandr19/manual-python-geologia.svg?color=forestgreen">
        &nbsp;&nbsp;
        <!-- Release date -->
        <img src="https://img.shields.io/github/release-date/kevinalexandr19/manual-python-geologia?color=gold">
    </span>
    <br>
    <span>
        <!-- Perfil de LinkedIn -->
        <a target="_blank" href="https://www.linkedin.com/in/kevin-alexander-gomez/">
            <img src="https://img.shields.io/badge/-Kevin Alexander Gomez-5eba00?style=social&logo=linkedin">
        </a>
        &nbsp;&nbsp;
        <!-- Perfil de Github -->
        <a target="_blank" href="https://github.com/kevinalexandr19">
            <img src="https://img.shields.io/github/followers/kevinalexandr19.svg?style=social&label=kevinalexandr19&maxAge=2592000">
        </a>
    </span>
    <br>
</div>

***

<span style="color:lightgreen; font-size:25px">**PG303 - Análisis geoespacial** </span>

Bienvenido al curso!!!

Vamos a revisar algunas aplicaciones del <span style="color:gold">análisis geoespacial</span> usando código en Python. <br>
Es necesario que tengas un conocimiento previo en programación con Python, estadística y sistemas de información geográfica.


<span style="color:gold; font-size:20px">**GeoPandas**</span>

***
- [¿Qué es GeoPandas?](#parte-1)
- [GeoDataFrame y GeoSeries](#parte-2)
- [Abrir un shapefile](#parte-3)
- [Visualizar los países](#parte-4)
- [Crear y guardar un shapefile](#parte-5)
- [Medir con el CRS adecuado](#parte-6)
- [Crear puntos a partir de una tabla](#parte-7)

***

<a id="parte-1"></a>

### <span style="color:lightgreen">**¿Qué es GeoPandas?**</span>
***

<span style="color:gold">GeoPandas</span> extiende las tablas de Pandas con una columna de **geometría** y un **sistema de referencia de coordenadas** (CRS). Las operaciones espaciales se realizan con geometrías de Shapely. Podemos usarlo para representar muestras, unidades litológicas y epicentros, calcular medidas o combinar capas. Para las matrices ráster usaremos Rioxarray en el siguiente cuaderno.

En JupyterLite la primera celda carga GeoPandas y el lector de archivos vectoriales Fiona. En Python local puedes preparar el entorno con `python -m pip install geopandas fiona matplotlib openpyxl "folium==0.18.0" "mapclassify==2.8.1"`.

In [ ]:
import sys

if sys.platform == "emscripten":
    import piplite
    from pyodide_js import loadPackage

    await loadPackage(["geopandas", "fiona", "matplotlib", "scikit-learn", "networkx",
                       "requests", "ssl", "jinja2", "xyzservices"])
    await piplite.install(["openpyxl==3.1.5", "folium==0.18.0", "branca==0.8.1",
                           "mapclassify==2.8.1"])

<a id="parte-2"></a>

### <span style="color:lightgreen">**GeoDataFrame y GeoSeries**</span>
***

Un `GeoDataFrame` combina atributos como el nombre y la población con una columna activa de geometría. Cuando seleccionas `gdf.geometry`, obtienes una `GeoSeries`. Los métodos espaciales del `GeoDataFrame` trabajan sobre esa columna activa.

| Elemento | Ejemplo | Pregunta antes de usarlo |
| --- | --- | --- |
| Atributo | Nombre del país | ¿Cómo identifico la fila que busco? |
| Geometría | Polígono o multipolígono | ¿Qué representa su contorno? |
| CRS | EPSG:4326 | ¿En qué sistema están x e y? |

<a id="parte-3"></a>

### <span style="color:lightgreen">**Abrir un shapefile**</span>
***

Conservamos el archivo `countries` del curso. Un shapefile usa varios archivos con el mismo nombre base. El `.shp` contiene geometrías, el `.dbf` los atributos, el `.shx` el índice y el `.prj` el sistema de coordenadas. La celda abre el `.shp` y Fiona encuentra los archivos asociados en la misma carpeta.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from pyproj import Geod

gdf = gpd.read_file("files/countries/countries.shp", engine="fiona")
print(f"{len(gdf)} geometrías | CRS: {gdf.crs} | Columna activa: {gdf.geometry.name}")
print(gdf.geom_type.value_counts().to_string())
gdf[["name", "continent", "pop_est", "geometry"]].head()

<a id="parte-4"></a>

### <span style="color:lightgreen">**Visualizar los países**</span>
***

`plot` dibuja la geometría activa. Primero vemos todo el archivo y luego ampliamos la región de Sudamérica con los límites de los ejes. Cambiar `xlim` e `ylim` modifica la vista. Para seleccionar las filas según el atributo `continent` usamos una condición en la tabla.

In [ ]:
sudamerica = gdf.loc[gdf["continent"] == "South America"].copy()

fig, ejes = plt.subplots(1, 2, figsize=(10, 4.1), layout="constrained")
gdf.plot(ax=ejes[0], color="#DEE8E9", edgecolor="#6B858C", linewidth=.3)
sudamerica.plot(ax=ejes[0], color="#78B7AD", edgecolor="#236559", linewidth=.3)
ejes[0].set(title="Países del archivo", xlabel="Longitud (°)", ylabel="Latitud (°)")
gdf.plot(ax=ejes[1], color="#DEE8E9", edgecolor="#6B858C", linewidth=.5)
sudamerica.plot(ax=ejes[1], color="#78B7AD", edgecolor="#236559", linewidth=.5)
ejes[1].set(xlim=(-90, -30), ylim=(-60, 15), title="Vista de Sudamérica",
            xlabel="Longitud (°)", ylabel="Latitud (°)")
for ax in ejes:
    ax.grid(alpha=.18)
    ax.set_axisbelow(True)
plt.show()

Con `explore` puedes mover el mapa, acercarte y consultar los atributos. La columna `pop_est` es una estimación del archivo y se usa para el color del polígono. Pasa sobre un país y verifica si el valor coincide con la tabla. Las geometrías salen del archivo local. La vista interactiva carga Leaflet en el navegador.

In [ ]:
sudamerica.explore(
    column="pop_est", cmap="YlGnBu", tiles=None, tooltip=["name", "pop_est"],
    legend_kwds={"caption": "Población estimada en el archivo"},
    style_kwds={"color": "#31444F", "weight": 1},
)

<a id="parte-5"></a>

### <span style="color:lightgreen">**Crear y guardar un shapefile**</span>
***

`to_file` escribe las filas seleccionadas y su geometría activa. Guardaremos la selección en una carpeta de salidas, con un nombre de archivo que termina en `.shp`. Abrirlo de nuevo permite comprobar el número de filas y el CRS.

In [ ]:
salidas = Path("salidas_pg303")
salidas.mkdir(exist_ok=True)
ruta = salidas / "sudamerica.shp"
sudamerica.to_file(ruta, driver="ESRI Shapefile", engine="fiona", index=False)
reabierto = gpd.read_file(ruta, engine="fiona")
assert len(reabierto) == len(sudamerica) and reabierto.crs == sudamerica.crs
print(f"{ruta} · {len(reabierto)} geometrías · {reabierto.crs}")

<a id="parte-6"></a>

### <span style="color:lightgreen">**Medir con el CRS adecuado**</span>
***

Aquí las coordenadas están en **grados**. Un grado de longitud representa distancias diferentes según la latitud. Para obtener áreas comparables en Sudamérica transformamos los polígonos a **Equal Earth (EPSG:8857)**, una proyección de áreas equivalentes. `to_crs` cambia las coordenadas. `set_crs` asigna una etiqueta al sistema en el que ya estaban las coordenadas.

Calculamos el área en el plano de Equal Earth. Para el perímetro usamos el elipsoide WGS84 con `Geod`, porque preservar el área de las figuras no garantiza preservar distancias. El centroide depende de la proyección y puede caer fuera de un polígono cóncavo o de una isla.

In [ ]:
peru = sudamerica.loc[sudamerica["iso_a3"] == "PER"]
peru_proyectado = peru.to_crs("EPSG:8857")
print("Extensión de Perú en grados:", peru.total_bounds.round(3))
print("Extensión de Perú en metros:", peru_proyectado.total_bounds.round(0))

region_m = sudamerica.to_crs("EPSG:8857")
geo = Geod(ellps="WGS84")
limites_geodesicos = sudamerica.geometry.boundary.map(geo.geometry_length)
medidas = sudamerica[["name"]].assign(
    area_km2=region_m.area / 1_000_000,
    perimetro_km=limites_geodesicos / 1_000,
)
centroides = region_m.centroid
medidas.sort_values("area_km2", ascending=False).round(1).head()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6), layout="constrained")
region_m.plot(ax=ax, color="#BFE4DC", edgecolor="#37595E", linewidth=.7)
centroides.plot(ax=ax, marker="+", markersize=65, color="#A74619", label="Centroides")
ax.set(title="Sudamérica en Equal Earth", xlabel="X (m)", ylabel="Y (m)")
ax.ticklabel_format(style="plain", useOffset=False)
ax.legend()
ax.grid(alpha=.15)
plt.show()

<a id="parte-7"></a>

### <span style="color:lightgreen">**Crear puntos a partir de una tabla**</span>
***

Conservamos el archivo de sismos del curso. `points_from_xy` recibe longitud primero y latitud después. Como sabemos que esas columnas están en WGS84, asignamos EPSG:4326 al construir el `GeoDataFrame`. En el mapa recortamos la ventana a Perú y alrededores. El criterio de magnitud se aplica a la tabla antes de dibujar.

In [ ]:
tabla_sismos = pd.read_excel("files/sismos.xlsx", engine="openpyxl")
sismos = tabla_sismos.rename(columns={"latitud (º)": "latitud", "longitud (º)": "longitud",
                                      "magnitud (M)": "magnitud", "profundidad (km)": "profundidad"})
sismos = sismos.dropna(subset=["longitud", "latitud", "magnitud"]).copy()
coor_validas = sismos["longitud"].between(-180, 180) & sismos["latitud"].between(-90, 90)
sismos = sismos.loc[coor_validas]
geometria = gpd.points_from_xy(sismos["longitud"], sismos["latitud"])
sismos_geo = gpd.GeoDataFrame(sismos, geometry=geometria, crs="EPSG:4326")
fuertes = sismos_geo.loc[sismos_geo["magnitud"] >= 6].copy()
print(f"{len(fuertes)} sismos con M ≥ 6 de {len(sismos_geo)} registros con coordenadas y magnitud")
fuertes[["longitud", "latitud", "magnitud", "geometry"]].head()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6), layout="constrained")
gdf.plot(ax=ax, color="#E5EFED", edgecolor="#5B747C", linewidth=.7)
fuertes.plot(ax=ax, color="#B84D12", alpha=.72, markersize=18, label="Epicentros · M ≥ 6")
ax.set(xlim=(-83, -68), ylim=(-19, 1), title="Sismos del archivo · Perú y alrededores",
       xlabel="Longitud (°)", ylabel="Latitud (°)")
ax.legend(loc="upper right")
ax.grid(alpha=.22)
plt.show()

# GeoJSON facilita compartir la selección en un solo archivo.
fuertes[["magnitud", "profundidad", "geometry"]].to_file(
    salidas / "sismos_m6.geojson", driver="GeoJSON", engine="fiona", index=False,
)

**Comprueba tu lectura.** En la vista aparecen epicentros mar adentro. Explica qué pasaría si usaras solo los puntos dentro del polígono de Perú. Cambia el umbral de magnitud y compara el número de filas con el número de puntos que aparecen en la ventana.

[Proyecciones en GeoPandas](https://geopandas.org/en/stable/docs/user_guide/projections.html) · [Mapas interactivos con explore](https://geopandas.org/en/stable/docs/user_guide/interactive_mapping.html)